# Third-order IETS of a spin-1/2

This example compares second- and third-order tunneling spectra using thermal populations. We then vary the magnetic field and potential scattering in a benchmark inspired by Ternes (2015), Figure 7(b,d).

The calculation describes weak-coupling scattering, not the strong-coupling Kondo regime.

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np

from spinfinity import SpinSys, bath, dIdV, dIdV3, hamil

## Setting up the spin Hamiltonian

For a magnetic field along z, Spinfinity uses

$$
H=-g\mu_B B S_z.
$$

The excitation energy is $\Delta E=g\mu_B B$. At 5 T and $g=2$, the tunneling thresholds are approximately $\pm0.579$ mV.

We use unpolarized reservoirs and thermal populations. `J` is the dimensionless substrate exchange, not the spin-spin interaction parameter `Jvector`. The conductance scale and lifetime broadening below are illustrative choices.

In [ ]:
spin = SpinSys([0.5])

# Hamiltonian parameters
spin.gfactorvector = [2.0]
spin.B = [0.0, 0.0, 5.0]  # T
spin.BTip = [0.0, 0.0, 0.0]

# Thermal populations and unpolarized reservoirs
spin.T = 1.0  # K
spin.TipPolarization = [0.0, 0.0, 0.0]
spin.SamplePolarization = [0.0, 0.0, 0.0]

# Scattering parameters
spin.G_st = 1e-9  # S; illustrative conductance scale
spin.G_ss = np.array([1e-8])  # S
spin.G_tt = 0
spin.b0 = 0.0
spin.U = np.array([0.0])

# Dimensionless exchange with the reservoirs, not Jvector in meV
spin.J = np.array([-0.05])
spin.J_tip = 0.0

# Bandwidth and numerical broadening
spin.w = 20.0  # meV
spin.lt = 0.005  # meV; chosen lifetime broadening

V_array = np.linspace(-2.0, 2.0, 401)  # mV
energy_step = 0.0025  # meV; internal integration spacing
fields = (0.0, 0.5, 1.0, 2.5, 5.0, 7.5, 10.0)  # T

hamil.calc_EigEnergies(spin, rebuildH=True)
bath.calc_Populations(spin, AllowPumping=False)

gap = spin.E_All[1] - spin.E_All[0]
print(f"Relative eigenenergies: {spin.E_All} meV")
print(f"Excitation thresholds: +/-{gap:.4f} mV")

## Comparing second and third order

Second order produces thermally broadened excitation steps. Third order adds interference involving virtual intermediate states, modifying the conductance around the excitation thresholds.

`calc_IETS_3rd` returns the combined second-plus-third-order spectrum. Both calculations below use thermal populations and `norm=False`.

In [ ]:
# Restore the baseline so this comparison can be rerun.
spin.B = [0.0, 0.0, 5.0]
spin.U[:] = 0.0
spin.J[:] = -0.05
hamil.calc_EigEnergies(spin, rebuildH=True)

V, conductance_2nd = dIdV.calc_IETS_2nd(spin, V_array=V_array, norm=False)
V, conductance_3rd = dIdV3.calc_IETS_3rd(
    spin, V_array=V_array, norm=False, energy_step=energy_step
)

gap = spin.E_All[1] - spin.E_All[0]
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)

ax.plot(V, conductance_2nd * 1e9, "k--", label="2nd order")
ax.plot(V, conductance_3rd * 1e9, label="2nd + 3rd order")
ax.axvline(-gap, color="gray", linestyle=":", linewidth=1)
ax.axvline(gap, color="gray", linestyle=":", linewidth=1)

ax.set(
    title="Thermal spin-1/2 spectrum at B = 5 T",
    xlabel="Bias voltage (mV)",
    ylabel="dI/dV (nS)",
)
ax.legend()
plt.show()

## Magnetic field and potential scattering

We compare field sweeps with `U=0` and `U=0.25`. The zero-field peak splits as the field increases. Potential scattering introduces bias asymmetry even though both reservoirs remain unpolarized.

For these literature-style panels, we display the dimensionless quantity

$$
\widetilde G(V)=M_{00}\frac{dI/dV}{G_{st}},
$$

where $M_{00}$ is the ground-state elastic squared scattering amplitude. With `b0=0`, this removes Spinfinity's elastic normalization from the displayed spectrum, allowing comparison across different values of U. The package calculation itself is unchanged.

The dashed curves show second order at 10 T. These panels compare spectral behavior, not a verified point-by-point reproduction of the published curves or the Scilab implementation.

In [ ]:
fig, axes = plt.subplots(
    1, 2, sharex=True, sharey=True, figsize=(12, 4), constrained_layout=True
)

for ax, potential in zip(axes, (0.0, 0.25), strict=True):
    spin.U[:] = potential

    for field in fields:
        spin.B = [0.0, 0.0, field]

        V, conductance = dIdV3.calc_IETS_3rd(
            spin,
            V_array=V_array,
            norm=False,
            energy_step=energy_step,
        )
        matrix_00 = bath.calc_TunnelingMatrixElements(spin, "st")[0, 0, 0]
        scaled_conductance = matrix_00 * conductance / spin.G_st

        ax.plot(V, scaled_conductance, label=f"B = {field:g} T")

    # The last field is 10 T: add the second-order reference at this field.
    V, second_order = dIdV.calc_IETS_2nd(spin, V_array=V_array, norm=False)
    matrix_00 = bath.calc_TunnelingMatrixElements(spin, "st")[0, 0, 0]
    ax.plot(
        V,
        matrix_00 * second_order / spin.G_st,
        "k--",
        linewidth=1.5,
        label="2nd order, B = 10 T",
    )

    ax.set(
        title=f"Potential scattering U = {potential:g}",
        xlabel="Bias voltage (mV)",
    )

axes[0].set_ylabel(r"$M_{00}(dI/dV)/G_{st}$")
axes[1].legend(fontsize=8, ncols=2)
plt.show()

# Restore the baseline and refresh derived quantities.
spin.B = [0.0, 0.0, 5.0]
spin.U[:] = 0.0
hamil.calc_EigEnergies(spin, rebuildH=True)
bath.calc_Populations(spin, AllowPumping=False)

## Checking numerical convergence

`energy_step` controls the internal integration grid, independently of the plotted voltage spacing. We halve it and compare the spectra.

This checks numerical convergence for the selected parameters. It does not establish agreement with the reference implementation.

In [ ]:
spin.B = [0.0, 0.0, 5.0]
spin.U[:] = 0.0
spin.J[:] = -0.05

_, coarse = dIdV3.calc_IETS_3rd(
    spin, V_array=V_array, norm=False, energy_step=energy_step
)
_, fine = dIdV3.calc_IETS_3rd(
    spin, V_array=V_array, norm=False, energy_step=energy_step / 2
)

relative_difference = np.max(np.abs(fine - coarse)) / np.max(np.abs(fine))
print(f"Maximum relative grid-refinement difference: {relative_difference:.2e}")

np.testing.assert_allclose(coarse, fine, rtol=1e-4, atol=spin.G_st * 1e-8)

## Reference

M. Ternes, *Spin excitations and correlations in scanning tunneling spectroscopy*, New Journal of Physics **17**, 063016 (2015). See Section 2.2 and Figure 7(b,d).

[DOI: 10.1088/1367-2630/17/6/063016](https://doi.org/10.1088/1367-2630/17/6/063016)